# Zgodność anotacji - stary prompt (Wykop, próba 1)

Pierwsza runda walidacji. Trzy osoby (Mikołaj, Patrycja, Zuza) niezależnie oznaczyły te same
150 wpisów z Wykopu etykietami `ai`, `work` i `stance`. Porównujemy je ze sobą i z modelem
`qwen3:8b` z **pierwotnym promptem** (`72f3a1da`), który oceniał wszystkie trzy etykiety naraz.

- `ai` - czy wpis dotyczy sztucznej inteligencji (`yes`/`no`),
- `work` - czy wpis dotyczy pracy, kariery lub rynku pracy (`yes`/`no`),
- `ai+work` - obie naraz; to filtr, który decyduje, co trafia do dalszej analizy,
- `stance` - co autor mówi o wpływie AI na pracę: `threat`, `reassurance`, `opportunity`,
  `mixed`, `none`.

Próba nie była czysto losowa: zawiera dużo wpisów, które model oznaczył jako `threat`.

In [ ]:
import sys
from itertools import combinations
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

sys.path.insert(0, "..")
from amc.classification.job_concern import PROMPT_HASH, YES_NO  # noqa: E402
from amc.classification.job_concern_details import STANCE  # noqa: E402

pd.set_option("display.max_colwidth", 200)
plt.rcParams["figure.figsize"] = (10, 4)

DATA = Path("../data/wykop")
KEY = ["type", "entry_id"]
SEED = 20261025
CATEGORIES = {"ai": YES_NO, "work": YES_NO, "ai+work": YES_NO, "stance": STANCE}

## Konfiguracja

In [ ]:
ANNOTATION_FILES = {
    "Mikołaj": DATA / "Anotacje_Wykop_Mikołaj.xlsx",
    "Patrycja": DATA / "Anotacje_Wykop_Patrycja.xlsx",
    "Zuza": DATA / "Anotacje_Wykop_Zuza.xlsx",
}
LABELS = ["ai", "work", "stance"]  # etykiety anotatorów
MODEL_FILE = DATA / "wykop_classified.csv"
MODEL_PROMPT = "72f3a1da"  # początek prompt_hash
MODEL_LABELS = ["ai", "work", "stance"]
BASELINE = None  # (plik, nazwa) - drugi model do porównania
OUT = DATA / "agreement" / "old_prompt"

## Wczytanie danych

In [ ]:
def clean(df, labels):
    """Etykiety małymi literami; wartości spoza słownika (np. puste) stają się brakiem."""
    df = df.reset_index(drop=True).copy()
    for label in labels:
        values = df[label].fillna("").astype(str).str.strip().str.lower()
        df[label] = values.where(values.isin(CATEGORIES[label]))
    if "ai" in labels and "work" in labels:
        both = df["ai"].eq("yes") & df["work"].eq("yes")
        df["ai+work"] = np.where(both, "yes", "no")
        df.loc[df["ai"].isna() | df["work"].isna(), "ai+work"] = np.nan
    return df


def load_annotations(path):
    df = pd.read_excel(path, sheet_name="sample", dtype=str).dropna(subset=["entry_id"])
    return clean(df, LABELS)


OUT.mkdir(parents=True, exist_ok=True)
sheets = {name: load_annotations(path) for name, path in ANNOTATION_FILES.items()}
ANNOTATORS = list(sheets)
base = sheets[ANNOTATORS[0]][[*KEY, "created_at", "url", "text"]]
for name, df in sheets.items():
    assert df[KEY].equals(base[KEY]), f"{name}: inne wpisy lub kolejność"
    missing = df[LABELS].isna().sum()
    if missing.any():
        print(f"{name}: brak poprawnej odpowiedzi -> {missing[missing > 0].to_dict()}")
print("wpisów:", len(base), "| anotatorzy:", ANNOTATORS)


def load_labels(path, labels, prompt=None):
    df = pd.read_csv(path, dtype=str, encoding="utf-8-sig")
    if prompt is not None:
        df = df[df["prompt_hash"].str.startswith(prompt)]
    df = base[KEY].merge(df, on=KEY, how="left", validate="one_to_one")
    assert df[labels[0]].notna().all(), f"{path.name}: brak etykiet dla części wpisów"
    return clean(df, labels)


ratings = dict(sheets)
model = load_labels(MODEL_FILE, MODEL_LABELS, MODEL_PROMPT)
ratings["model"] = model
print("model:", model["model"].dropna().unique(), "| prompt:", MODEL_PROMPT[:8])
if BASELINE is not None:
    ratings[BASELINE[1]] = load_labels(BASELINE[0], ["ai", "work"])
RATERS = list(ratings)
ALL_LABELS = [*LABELS[:2], "ai+work", *LABELS[2:]] if "work" in LABELS else LABELS


def raters_for(label):
    return [r for r in RATERS if label in ratings[r].columns]

## Rozkład etykiet

In [ ]:
pd.concat(
    {
        label: pd.DataFrame(
            {r: ratings[r][label].value_counts() for r in raters_for(label)}
        ).reindex(CATEGORIES[label])
        for label in ALL_LABELS
    }
).astype("Int64")

## Miary zgodności

- **zgodność %** - odsetek par ocen, które są takie same;
- **kappa Cohena** - dla pary oceniających (także model z człowiekiem), z poprawką na
  zgodność przypadkową wynikającą z rozkładu etykiet każdego z nich;
- **kappa Fleissa** - jedna liczba dla wszystkich anotatorów naraz.

Przedziały ufności 95% z bootstrapu po wpisach (2000 losowań). Wpisy z brakującą odpowiedzią
są pomijane tylko w porównaniach, których dotyczą.

In [ ]:
def labels_of(label, raters):
    """Macierz wpis x oceniający; wiersze z jakimkolwiek brakiem są pomijane."""
    values = pd.concat([ratings[r][label].rename(r) for r in raters], axis=1).dropna()
    return values.to_numpy(dtype=object), values.index


def percent_agreement(values):
    """Odsetek zgodnych par ocen; dla dwóch oceniających to odsetek identycznych ocen."""
    m = values.shape[1]
    counts = np.stack([(values == c).sum(axis=1) for c in np.unique(values)], axis=1)
    return (((counts**2).sum(axis=1) - m) / (m * (m - 1))).mean()


def cohen_kappa(values):
    a, b = values[:, 0], values[:, 1]
    p_o = np.mean(a == b)
    p_e = sum(np.mean(a == c) * np.mean(b == c) for c in np.union1d(a, b))
    return (p_o - p_e) / (1 - p_e) if p_e < 1 else np.nan


def fleiss_kappa(values):
    counts = np.stack([(values == c).sum(axis=1) for c in np.unique(values)], axis=1)
    p_e = ((counts.sum(axis=0) / counts.sum()) ** 2).sum()
    p_o = percent_agreement(values)
    return (p_o - p_e) / (1 - p_e) if p_e < 1 else np.nan


def bootstrap(statistic, values, iterations=2000, seed=SEED):
    rng = np.random.default_rng(seed)
    draws = [
        statistic(values[rng.integers(0, len(values), len(values))]) for _ in range(iterations)
    ]
    low, high = np.nanpercentile(draws, [2.5, 97.5])
    return f"[{low:.2f}, {high:.2f}]"


def landis_koch(value):
    """Opis słowny według Landisa i Kocha (1977)."""
    if np.isnan(value):
        return ""
    for limit, name in [(0, "brak"), (0.2, "niewielka"), (0.4, "dostateczna"),
                        (0.6, "umiarkowana"), (0.8, "znaczna")]:
        if value <= limit:
            return name
    return "prawie pełna"


# Kontrola wzorów: pełna zgodność daje 1, a oceny losowe i niezależne około 0
check = np.array([["yes", "yes"], ["no", "no"], ["yes", "yes"], ["no", "no"]], dtype=object)
assert np.isclose(cohen_kappa(check), 1) and np.isclose(fleiss_kappa(check), 1)
noise = np.random.default_rng(1).choice(["yes", "no"], size=(20000, 2)).astype(object)
assert abs(cohen_kappa(noise)) < 0.03 and abs(fleiss_kappa(noise)) < 0.03

## 1. Zgodność między anotatorami

In [ ]:
rows = []
for label in ALL_LABELS:
    values, _ = labels_of(label, ANNOTATORS)
    kappa = fleiss_kappa(values)
    rows.append({
        "etykieta": label,
        "wpisów": len(values),
        "zgodność %": percent_agreement(values),
        "kappa Fleissa": kappa,
        "95% CI": bootstrap(fleiss_kappa, values),
        "ocena": landis_koch(kappa),
    })
humans = pd.DataFrame(rows).set_index("etykieta")
humans.to_csv(OUT / "agreement_annotators.csv", encoding="utf-8-sig")
humans.round(3)

## 2. Kappa Cohena dla każdej pary oceniających

In [ ]:
pairs = []
for label in ALL_LABELS:
    for a, b in combinations(raters_for(label), 2):
        values, _ = labels_of(label, [a, b])
        pairs.append({
            "etykieta": label,
            "para": f"{a} - {b}",
            "rodzaj": "człowiek - człowiek" if {a, b} <= set(ANNOTATORS) else "z modelem",
            "wpisów": len(values),
            "zgodność %": np.mean(values[:, 0] == values[:, 1]),
            "kappa Cohena": cohen_kappa(values),
            "95% CI": bootstrap(cohen_kappa, values),
        })
pairs = pd.DataFrame(pairs)
pairs.to_csv(OUT / "cohen_pairs.csv", index=False, encoding="utf-8-sig")
pairs.pivot(index="para", columns="etykieta", values="kappa Cohena").round(3)

In [ ]:
fig, axes = plt.subplots(1, len(ALL_LABELS), figsize=(4 * len(ALL_LABELS), 3.5))
for ax, label in zip(np.atleast_1d(axes), ALL_LABELS):
    part = pairs[pairs["etykieta"] == label]
    colors = np.where(part["rodzaj"] == "z modelem", "tab:orange", "tab:blue")
    ax.barh(part["para"], part["kappa Cohena"], color=colors)
    ax.axvline(0.6, color="grey", ls=":", lw=1)
    ax.set_title(label)
    ax.set_xlim(min(0, part["kappa Cohena"].min() - 0.05), 1)
fig.suptitle("Kappa Cohena: niebieskie - ludzie, pomarańczowe - model (linia: 0,6)")
plt.tight_layout()
plt.savefig(OUT / "cohen_pairs.png", dpi=120)
plt.show()

## 3. Model a etykieta złota

In [ ]:
def gold_label(label):
    """Odpowiedź co najmniej dwojga anotatorów; przy dwóch - tylko gdy oboje się zgadzają."""
    votes = pd.concat([ratings[a][label] for a in ANNOTATORS], axis=1)
    result = []
    for row in votes.itertuples(index=False):
        counts = pd.Series(row).dropna().value_counts()
        top = counts.index[0] if len(counts) else None
        ok = len(counts) and counts.iloc[0] >= 2 and (len(counts) == 1 or counts.iloc[1] < counts.iloc[0])
        result.append(top if ok else np.nan)
    return pd.Series(result, dtype=object)


gold = pd.DataFrame({label: gold_label(label) for label in ALL_LABELS})
ratings["złota"] = gold
pd.DataFrame({
    "z etykietą złotą": gold.notna().sum(),
    "bez (spór lub brak)": gold.isna().sum(),
})

In [ ]:
def per_class(truth, predicted, categories):
    rows = []
    for c in categories:
        tp = np.sum((predicted == c) & (truth == c))
        fp = np.sum((predicted == c) & (truth != c))
        fn = np.sum((predicted != c) & (truth == c))
        precision = tp / (tp + fp) if tp + fp else np.nan
        recall = tp / (tp + fn) if tp + fn else np.nan
        f1 = 2 * precision * recall / (precision + recall) if precision + recall else np.nan
        rows.append({"kategoria": c, "złota": tp + fn, "model": tp + fp,
                     "precyzja": precision, "czułość": recall, "F1": f1})
    return pd.DataFrame(rows).set_index("kategoria")


MODELS = [r for r in RATERS if r not in ANNOTATORS]
human_mean = pairs[pairs["rodzaj"] == "człowiek - człowiek"].groupby("etykieta")["kappa Cohena"].mean()
rows = []
for name in MODELS:
    for label in [lab for lab in ALL_LABELS if lab in ratings[name].columns]:
        values, _ = labels_of(label, ["złota", name])
        kappa = cohen_kappa(values)
        yes = per_class(values[:, 0], values[:, 1], ["yes"]).iloc[0] if label != "stance" else None
        rows.append({
            "model": name,
            "etykieta": label,
            "wpisów": len(values),
            "trafność": np.mean(values[:, 0] == values[:, 1]),
            "kappa Cohena": kappa,
            "95% CI": bootstrap(cohen_kappa, values),
            "ocena": landis_koch(kappa),
            "precyzja yes": yes["precyzja"] if yes is not None else np.nan,
            "czułość yes": yes["czułość"] if yes is not None else np.nan,
            "kappa ludzi (średnia par)": human_mean.get(label, np.nan),
        })
vs_gold = pd.DataFrame(rows).set_index(["model", "etykieta"])
vs_gold.to_csv(OUT / "model_vs_gold.csv", encoding="utf-8-sig")
vs_gold.round(3)

### Macierze pomyłek i miary per kategoria

In [ ]:
for name in MODELS:
    for label in [lab for lab in ALL_LABELS if lab in ratings[name].columns]:
        values, _ = labels_of(label, ["złota", name])
        cats = CATEGORIES[label]
        print(f"\n=== {name}: {label} (wiersze: złota, kolumny: {name})")
        display(pd.crosstab(
            pd.Categorical(values[:, 0], cats), pd.Categorical(values[:, 1], cats),
            rownames=["złota"], colnames=[name], dropna=False,
        ))
        scores = per_class(values[:, 0], values[:, 1], cats)
        scores.to_csv(OUT / f"per_class_{name}_{label}.csv".replace(" ", "_"), encoding="utf-8-sig")
        display(scores.round(3))

## 4. Rozbieżności do omówienia

Wpisy, w których model różni się od etykiety złotej, oraz wpisy, przy których anotatorzy się
różnili. Zapisujemy je do folderu wyników.

In [ ]:
def show(mask, labels):
    columns = {}
    for label in labels:
        for rater in [*raters_for(label), "złota"]:
            if label in ratings[rater].columns:
                columns[f"{label}: {rater}"] = ratings[rater][label]
    view = pd.DataFrame({
        "url": base["url"],
        "text": base["text"].str.replace("\n", " ").str.slice(0, 300),
        **columns,
    })
    return view[mask.to_numpy()]


model_labels = [lab for lab in ALL_LABELS if lab in model.columns]
model_split = pd.Series(False, index=base.index)
for label in model_labels:
    model_split |= gold[label].notna() & model[label].ne(gold[label])
human_split = pd.Series(False, index=base.index)
for label in LABELS:
    votes = pd.concat([ratings[a][label] for a in ANNOTATORS], axis=1)
    human_split |= votes.nunique(axis=1, dropna=False) > 1
print("model inaczej niż złota:", model_split.sum(), "| anotatorzy niezgodni:", human_split.sum())

show(model_split, model_labels).to_csv(OUT / "model_vs_gold_rows.csv", encoding="utf-8-sig")
show(human_split, LABELS).to_csv(OUT / "annotators_split.csv", encoding="utf-8-sig")
show(model_split, model_labels)

In [ ]:
show(human_split, LABELS)

## Jak czytać wyniki

- **Ludzie między sobą.** Kappa powyżej 0,8 to zgodność prawie pełna, 0,6-0,8 znaczna
  (Landis i Koch 1977). Jeśli jest niska, problemem jest definicja etykiety, nie model -
  najpierw doprecyzujcie wytyczne na podstawie listy rozbieżności.
- **Model.** Jeśli kappa model-człowiek mieści się w zakresie kapp człowiek-człowiek, model
  może zastąpić anotatora. Jeśli jest wyraźnie niższa, trzeba poprawić prompt albo model.
- **`ai+work`** to filtr do dalszej analizy: precyzja mówi, ile przepuszczonych wpisów jest
  zbędnych, czułość - ile właściwych wpisów odpada.
- Kappa przy rzadkich kategoriach bywa niska mimo wysokiej zgodności procentowej - wtedy
  patrz też na liczby w macierzy pomyłek.